In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="multimodal",
    notebook_name="01_vision_language.ipynb"
)

# Vision-Language Models: Connecting Images and Text

In 2021, CLIP showed that a model trained only to match images with captions could classify images into categories it had never seen. No labeled training data for those categories. No fine-tuning. Just compare embeddings.

In this notebook, we will build the core pieces of CLIP from scratch:

1. **Cosine similarity** — how to measure if two embeddings point in the same direction
2. **Contrastive learning** — how to train a model to push matching pairs together and non-matching pairs apart
3. **Embedding space visualization** — see image-text alignment in 2D
4. **Zero-shot classification** — classify without labeled examples
5. **Temperature scaling** — how one number controls the entire training dynamic

**Prerequisites:** NumPy basics, what an embedding is ([multimodal README](../README.md)), what attention does ([01-transformers](../../01-transformers/architecture/attention-mechanisms.md)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['font.size'] = 12

np.random.seed(42)
print("Setup complete!")

## Part 1: Cosine Similarity — Measuring Direction, Not Size

Before we can train a vision-language model, we need a way to measure how similar two embeddings are.

We could use the dot product, but there is a problem: longer vectors give bigger dot products even when they point in the same direction. We want a measure that only cares about **direction**, not length.

**Cosine similarity** divides the dot product by the lengths of both vectors. The result is always between -1 and +1:
- **+1** = same direction (perfect match)
- **0** = perpendicular (no relationship)
- **-1** = opposite direction

In [ ]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)


# Example: three vectors
v_dog = np.array([0.9, 0.1, 0.8, 0.2])    # "dog" embedding
v_puppy = np.array([0.85, 0.15, 0.75, 0.25])  # "puppy" embedding (similar to dog)
v_car = np.array([-0.5, 0.9, -0.3, 0.7])   # "car" embedding (different)

print("Cosine similarities:")
print(f"  dog vs puppy:  {cosine_similarity(v_dog, v_puppy):.4f}  (high — similar concepts)")
print(f"  dog vs car:    {cosine_similarity(v_dog, v_car):.4f}  (low — different concepts)")
print(f"  puppy vs car:  {cosine_similarity(v_puppy, v_car):.4f}  (low — different concepts)")

# Show that scaling a vector does NOT change its cosine similarity
v_dog_big = v_dog * 100
print(f"\n  dog vs dog*100: {cosine_similarity(v_dog, v_dog_big):.4f}  (length does not matter!)")
print(f"  ||dog||   = {np.linalg.norm(v_dog):.2f}")
print(f"  ||dog*100|| = {np.linalg.norm(v_dog_big):.2f}")
print("  Same direction, different length → cosine similarity = 1.0")

### Visualizing Cosine Similarity in 2D

Let's see what cosine similarity looks like with 2D vectors. We will plot several vectors and show their cosine similarities.

In [ ]:
# 2D vectors for visualization
vectors = {
    '"dog" (image)': np.array([0.8, 0.6]),
    '"dog" (text)': np.array([0.75, 0.65]),
    '"cat" (image)': np.array([0.3, 0.95]),
    '"car" (text)': np.array([-0.7, 0.7]),
}

colors = ['#2196F3', '#1976D2', '#4CAF50', '#F44336']

fig, ax = plt.subplots(figsize=(8, 8))

for (name, vec), color in zip(vectors.items(), colors):
    # Normalize for display
    vec_norm = vec / np.linalg.norm(vec)
    ax.annotate('', xy=vec_norm, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(vec_norm[0]*1.1, vec_norm[1]*1.1, name, fontsize=11,
            color=color, fontweight='bold', ha='center')

# Draw unit circle
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.2, lw=1)

ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_aspect('equal')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_title('Cosine Similarity = Direction Agreement\n(closer arrows = higher similarity)', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print pairwise similarities
names = list(vectors.keys())
vecs = [vectors[n] for n in names]
print("Pairwise cosine similarities:")
for i in range(len(names)):
    for j in range(i+1, len(names)):
        sim = cosine_similarity(vecs[i], vecs[j])
        print(f"  {names[i]:20s} vs {names[j]:20s} = {sim:.4f}")

## Part 2: Contrastive Learning — The Training Loop

Now we build the core of CLIP: **contrastive learning**.

The idea is simple:
1. We have a batch of N image-text pairs (image₁ matches text₁, image₂ matches text₂, etc.)
2. Compute cosine similarity between every image and every text → an N×N matrix
3. The diagonal entries (matching pairs) should be high
4. The off-diagonal entries (non-matching pairs) should be low

We use the **InfoNCE loss** (also called NT-Xent) to train this.

### Setting Up Synthetic Data

We will not use real images and text. Instead, we will create synthetic embeddings that simulate what image and text encoders would produce. This lets us focus on the contrastive learning mechanics.

In [ ]:
# Create synthetic "concepts" — each concept has a ground-truth direction
# Images and text for the same concept should point in similar directions

n_concepts = 8
embed_dim = 16

# Ground truth concept directions (what the model should learn)
np.random.seed(42)
concept_directions = np.random.randn(n_concepts, embed_dim)
concept_directions = concept_directions / np.linalg.norm(concept_directions, axis=1, keepdims=True)

concept_names = ["dog", "cat", "car", "sunset", "pizza", "mountain", "book", "guitar"]

def make_batch(batch_size, noise_level=0.5):
    """Create a batch of noisy image and text embeddings.
    
    Image i and text i represent the same concept (matching pair).
    All other pairs are non-matching.
    """
    # Pick random concepts for this batch
    indices = np.random.choice(n_concepts, size=batch_size, replace=False)
    
    # Image embeddings = concept direction + noise
    image_embeds = concept_directions[indices] + np.random.randn(batch_size, embed_dim) * noise_level
    
    # Text embeddings = concept direction + different noise
    text_embeds = concept_directions[indices] + np.random.randn(batch_size, embed_dim) * noise_level
    
    return image_embeds, text_embeds, indices


# Demo: make a small batch
batch_images, batch_texts, batch_indices = make_batch(4)
print(f"Batch of {len(batch_indices)} image-text pairs:")
for i, idx in enumerate(batch_indices):
    print(f"  Pair {i}: image of '{concept_names[idx]}' <--> text '{concept_names[idx]}'")
print(f"\nImage embeddings shape: {batch_images.shape}")
print(f"Text embeddings shape:  {batch_texts.shape}")

### Computing the Similarity Matrix

The heart of contrastive learning: compute cosine similarity between **every** image embedding and **every** text embedding. This gives us an N×N matrix.

The diagonal should be high (matching pairs). Everything else should be low.

In [ ]:
def compute_similarity_matrix(image_embeds, text_embeds):
    """Compute N×N cosine similarity matrix between image and text embeddings."""
    # L2 normalize embeddings
    image_norm = image_embeds / np.linalg.norm(image_embeds, axis=1, keepdims=True)
    text_norm = text_embeds / np.linalg.norm(text_embeds, axis=1, keepdims=True)
    
    # Cosine similarity = dot product of normalized vectors
    # (N, d) @ (d, N) = (N, N)
    similarity = image_norm @ text_norm.T
    return similarity


# Compute similarity matrix for our batch
sim_matrix = compute_similarity_matrix(batch_images, batch_texts)
print(f"Similarity matrix shape: {sim_matrix.shape}")
print(f"  Each entry (i,j) = cosine_sim(image_i, text_j)\n")

# Display the matrix
batch_names = [concept_names[idx] for idx in batch_indices]
print(f"{'':>12}", end="")
for name in batch_names:
    print(f"T:{name:>8}", end="")
print()
print(f"{'':>12}" + "-" * (10 * len(batch_names)))
for i, name in enumerate(batch_names):
    print(f"I:{name:>8} |", end="")
    for j in range(len(batch_names)):
        marker = " ←" if i == j else "  "
        print(f"{sim_matrix[i, j]:>8.3f}{marker}", end="")
    print()
print("\n← marks the diagonal (matching pairs — these should be highest)")

In [ ]:
# Visualize the similarity matrix
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)

ax.set_xticks(range(len(batch_names)))
ax.set_yticks(range(len(batch_names)))
ax.set_xticklabels([f'Text:\n{n}' for n in batch_names], fontsize=11)
ax.set_yticklabels([f'Image:\n{n}' for n in batch_names], fontsize=11)
ax.set_title('Cosine Similarity Matrix (before training)\nDiagonal = matching pairs', fontsize=13)

for i in range(len(batch_names)):
    for j in range(len(batch_names)):
        color = 'white' if abs(sim_matrix[i, j]) > 0.5 else 'black'
        weight = 'bold' if i == j else 'normal'
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center',
                fontsize=12, color=color, fontweight=weight)

plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.tight_layout()
plt.show()

### The InfoNCE Loss

Now we implement the actual loss function. The loss has two parts:

1. **Image-to-text:** For each image, softmax over all texts. The correct text should get the highest probability.
2. **Text-to-image:** For each text, softmax over all images. The correct image should get the highest probability.

We divide all similarities by a temperature τ before softmax. This controls how sharp the distribution is.

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def infonce_loss(image_embeds, text_embeds, temperature=0.07):
    """Compute the symmetric InfoNCE (NT-Xent) contrastive loss.
    
    This is the loss function used by CLIP.
    
    Args:
        image_embeds: (N, d) image embeddings
        text_embeds:  (N, d) text embeddings  
        temperature:  scalar that controls softmax sharpness
    
    Returns:
        total_loss: scalar
        loss_i2t:   image-to-text loss
        loss_t2i:   text-to-image loss
    """
    N = len(image_embeds)
    
    # Step 1: Compute similarity matrix
    sim = compute_similarity_matrix(image_embeds, text_embeds)
    
    # Step 2: Scale by temperature
    logits = sim / temperature
    
    # Step 3: Labels = diagonal indices (image_i matches text_i)
    labels = np.arange(N)
    
    # Step 4: Image-to-text loss (softmax along rows)
    probs_i2t = softmax(logits, axis=1)  # Each row sums to 1
    loss_i2t = -np.mean(np.log(probs_i2t[np.arange(N), labels] + 1e-8))
    
    # Step 5: Text-to-image loss (softmax along columns)
    probs_t2i = softmax(logits, axis=0)  # Each column sums to 1
    loss_t2i = -np.mean(np.log(probs_t2i[labels, np.arange(N)] + 1e-8))
    
    # Step 6: Total loss
    total_loss = (loss_i2t + loss_t2i) / 2
    
    return total_loss, loss_i2t, loss_t2i


# Compute loss on our batch
loss, loss_i2t, loss_t2i = infonce_loss(batch_images, batch_texts, temperature=0.07)
print(f"InfoNCE Loss:")
print(f"  Image-to-text loss: {loss_i2t:.4f}")
print(f"  Text-to-image loss: {loss_t2i:.4f}")
print(f"  Total loss:         {loss:.4f}")
print(f"\n  (Lower is better. Perfect = 0.0, random = log({len(batch_images)}) = {np.log(len(batch_images)):.4f})")

### Training a Simplified CLIP

Now we put it all together. We will train simple linear projection layers (simulating the image and text encoders) to align embeddings using contrastive learning.

In real CLIP, the encoders are a ViT and a Transformer. Here, we use small linear layers so the training runs in seconds without a GPU.

In [ ]:
# Simple linear encoders (simulating image and text encoders)
# In real CLIP, these would be ViT and Transformer

input_dim = embed_dim  # 16
output_dim = 8         # Project to smaller shared space

np.random.seed(0)

# Learnable projection matrices
W_image = np.random.randn(input_dim, output_dim) * 0.1
W_text = np.random.randn(input_dim, output_dim) * 0.1

def encode_image(raw_embeds, W):
    """Project raw image embeddings through the image encoder."""
    projected = raw_embeds @ W
    # L2 normalize
    return projected / np.linalg.norm(projected, axis=1, keepdims=True)

def encode_text(raw_embeds, W):
    """Project raw text embeddings through the text encoder."""
    projected = raw_embeds @ W
    return projected / np.linalg.norm(projected, axis=1, keepdims=True)


# Training loop with numerical gradient descent
learning_rate = 0.5
temperature = 0.1
batch_size = 6
n_steps = 200
losses = []

print("Training a simplified CLIP model...")
print(f"  Embedding dim: {input_dim} → {output_dim}")
print(f"  Batch size: {batch_size}")
print(f"  Temperature: {temperature}")
print(f"  Steps: {n_steps}\n")

for step in range(n_steps):
    # Get a batch of raw embeddings
    raw_images, raw_texts, _ = make_batch(batch_size, noise_level=0.3)
    
    # Forward pass
    img_enc = encode_image(raw_images, W_image)
    txt_enc = encode_text(raw_texts, W_text)
    
    loss, _, _ = infonce_loss(img_enc, txt_enc, temperature=temperature)
    losses.append(loss)
    
    # Numerical gradient for W_image
    eps = 1e-4
    grad_W_image = np.zeros_like(W_image)
    for i in range(W_image.shape[0]):
        for j in range(W_image.shape[1]):
            W_image[i, j] += eps
            img_enc_p = encode_image(raw_images, W_image)
            loss_p, _, _ = infonce_loss(img_enc_p, txt_enc, temperature=temperature)
            W_image[i, j] -= 2 * eps
            img_enc_m = encode_image(raw_images, W_image)
            loss_m, _, _ = infonce_loss(img_enc_m, txt_enc, temperature=temperature)
            W_image[i, j] += eps  # restore
            grad_W_image[i, j] = (loss_p - loss_m) / (2 * eps)
    
    # Numerical gradient for W_text
    grad_W_text = np.zeros_like(W_text)
    for i in range(W_text.shape[0]):
        for j in range(W_text.shape[1]):
            W_text[i, j] += eps
            txt_enc_p = encode_text(raw_texts, W_text)
            loss_p, _, _ = infonce_loss(img_enc, txt_enc_p, temperature=temperature)
            W_text[i, j] -= 2 * eps
            txt_enc_m = encode_text(raw_texts, W_text)
            loss_m, _, _ = infonce_loss(img_enc, txt_enc_m, temperature=temperature)
            W_text[i, j] += eps  # restore
            grad_W_text[i, j] = (loss_p - loss_m) / (2 * eps)
    
    # Update weights
    W_image -= learning_rate * grad_W_image
    W_text -= learning_rate * grad_W_text
    
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  Step {step:>3d}: loss = {loss:.4f}")

print(f"\nTraining complete!")
print(f"  Initial loss: {losses[0]:.4f}")
print(f"  Final loss:   {losses[-1]:.4f}")
print(f"  Random baseline: {np.log(batch_size):.4f}")

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, color='#2196F3', linewidth=1.5, alpha=0.7)

# Smoothed loss
window = 10
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, len(losses)), smoothed, color='#1565C0', linewidth=2.5, label='Smoothed')

ax.axhline(y=np.log(batch_size), color='red', linestyle='--', alpha=0.5, label=f'Random baseline (log {batch_size} = {np.log(batch_size):.2f})')
ax.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='Perfect (0.0)')

ax.set_xlabel('Training Step', fontsize=13)
ax.set_ylabel('InfoNCE Loss', fontsize=13)
ax.set_title('Contrastive Learning Training Curve', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The loss decreases from the random baseline toward zero.")
print("This means the model is learning to push matching pairs together.")

## Part 3: Visualizing the Embedding Space

Let's see what the trained model actually learned. We will encode all concepts and plot their image and text embeddings in 2D using PCA (a method that finds the two most important directions in high-dimensional data).

In [ ]:
# Encode all concepts (with low noise to see the structure)
all_images = concept_directions + np.random.randn(n_concepts, embed_dim) * 0.1
all_texts = concept_directions + np.random.randn(n_concepts, embed_dim) * 0.1

img_encoded = encode_image(all_images, W_image)
txt_encoded = encode_text(all_texts, W_text)

# Combine for PCA
all_embeds = np.vstack([img_encoded, txt_encoded])

# Simple PCA: project to 2D
mean = all_embeds.mean(axis=0)
centered = all_embeds - mean
_, _, Vt = np.linalg.svd(centered, full_matrices=False)
projected = centered @ Vt[:2].T

img_2d = projected[:n_concepts]
txt_2d = projected[n_concepts:]

# Plot
fig, ax = plt.subplots(figsize=(10, 8))

colors_concepts = plt.cm.Set2(np.linspace(0, 1, n_concepts))

for i in range(n_concepts):
    # Image embedding (circle)
    ax.scatter(img_2d[i, 0], img_2d[i, 1], c=[colors_concepts[i]], s=200,
               marker='o', edgecolors='black', linewidth=1.5, zorder=5)
    # Text embedding (triangle)
    ax.scatter(txt_2d[i, 0], txt_2d[i, 1], c=[colors_concepts[i]], s=200,
               marker='^', edgecolors='black', linewidth=1.5, zorder=5)
    # Draw line connecting matching pair
    ax.plot([img_2d[i, 0], txt_2d[i, 0]], [img_2d[i, 1], txt_2d[i, 1]],
            color=colors_concepts[i], linestyle='--', alpha=0.5, linewidth=1)
    # Label
    mid_x = (img_2d[i, 0] + txt_2d[i, 0]) / 2
    mid_y = (img_2d[i, 1] + txt_2d[i, 1]) / 2
    ax.annotate(concept_names[i], (mid_x, mid_y), fontsize=11, fontweight='bold',
                ha='center', va='bottom')

# Legend
ax.scatter([], [], c='gray', s=100, marker='o', edgecolors='black', label='Image embedding')
ax.scatter([], [], c='gray', s=100, marker='^', edgecolors='black', label='Text embedding')
ax.legend(fontsize=12, loc='upper right')

ax.set_title('Learned Embedding Space (PCA projection to 2D)\nMatching image-text pairs are connected by dashed lines', fontsize=13)
ax.set_xlabel('PCA Component 1', fontsize=12)
ax.set_ylabel('PCA Component 2', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print pairwise similarities for matching pairs
print("Cosine similarities for matching pairs (after training):")
for i in range(n_concepts):
    sim = np.dot(img_encoded[i], txt_encoded[i])
    print(f"  {concept_names[i]:>10s}: image-text similarity = {sim:.4f}")

## Part 4: Zero-Shot Classification

Now the exciting part. We can use the trained model to classify new images into categories — without any labeled training data for those categories.

The method:
1. Encode the image
2. Encode each category name as text (e.g., "a photo of a dog")
3. Compare the image embedding to each text embedding
4. The category with the highest similarity wins

In [ ]:
def zero_shot_classify(image_embed, category_text_embeds, category_names, temperature=0.1):
    """Classify an image into categories using cosine similarity.
    
    Args:
        image_embed: (d,) single image embedding (already normalized)
        category_text_embeds: (C, d) text embeddings for each category
        category_names: list of category name strings
        temperature: softmax temperature
    
    Returns:
        predicted_category, probabilities
    """
    # Cosine similarities (embeddings are already normalized)
    similarities = image_embed @ category_text_embeds.T  # (C,)
    
    # Convert to probabilities
    logits = similarities / temperature
    probs = softmax(logits)
    
    predicted_idx = np.argmax(probs)
    return category_names[predicted_idx], probs


# Encode all concept names as "category text embeddings"
category_text_raw = concept_directions  # Using ground truth directions as raw text
category_text_encoded = encode_text(category_text_raw, W_text)

# Test: classify each image
print("Zero-Shot Classification Results:")
print("=" * 60)

correct = 0
for i in range(n_concepts):
    # Create a noisy image for this concept
    test_image_raw = concept_directions[i] + np.random.randn(embed_dim) * 0.2
    test_image_encoded = encode_image(test_image_raw.reshape(1, -1), W_image)[0]
    
    predicted, probs = zero_shot_classify(
        test_image_encoded, category_text_encoded, concept_names
    )
    
    is_correct = predicted == concept_names[i]
    correct += is_correct
    status = "CORRECT" if is_correct else "WRONG"
    
    # Show top-3 predictions
    top3_idx = np.argsort(probs)[-3:][::-1]
    top3_str = ", ".join([f"{concept_names[j]} ({probs[j]:.1%})" for j in top3_idx])
    
    print(f"  Image: {concept_names[i]:>10s} → Predicted: {predicted:>10s}  [{status}]")
    print(f"{'':>28}Top-3: {top3_str}")

print(f"\nAccuracy: {correct}/{n_concepts} = {correct/n_concepts:.0%}")
print("\nThis is zero-shot classification — no labeled examples were used!")
print("The model learned to connect images and text through contrastive training alone.")

In [ ]:
# Visualize the classification as a similarity heatmap
test_sim_matrix = np.zeros((n_concepts, n_concepts))
for i in range(n_concepts):
    test_image_raw = concept_directions[i] + np.random.randn(embed_dim) * 0.1
    test_image_encoded = encode_image(test_image_raw.reshape(1, -1), W_image)[0]
    test_sim_matrix[i] = test_image_encoded @ category_text_encoded.T

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(test_sim_matrix, cmap='RdYlBu_r', vmin=-1, vmax=1)

ax.set_xticks(range(n_concepts))
ax.set_yticks(range(n_concepts))
ax.set_xticklabels(concept_names, fontsize=10, rotation=45, ha='right')
ax.set_yticklabels(concept_names, fontsize=10)
ax.set_xlabel('Text Category', fontsize=12)
ax.set_ylabel('Test Image', fontsize=12)
ax.set_title('Zero-Shot Classification Similarity Matrix\n(bright diagonal = correct classifications)', fontsize=13)

for i in range(n_concepts):
    for j in range(n_concepts):
        color = 'white' if abs(test_sim_matrix[i, j]) > 0.5 else 'black'
        ax.text(j, i, f'{test_sim_matrix[i, j]:.2f}', ha='center', va='center',
                fontsize=9, color=color)

plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.tight_layout()
plt.show()

print("A bright diagonal means the model correctly matches each image to its text category.")

## Part 5: Temperature Scaling

Temperature τ is one number that controls the entire training dynamic. Let's see exactly what it does.

When we divide similarities by τ before softmax:
- **Small τ (e.g., 0.01):** Similarities are amplified. Softmax becomes very sharp — nearly all probability goes to the highest similarity.
- **Large τ (e.g., 1.0):** Similarities are dampened. Softmax becomes flat — probability is spread across all options.

In [ ]:
# Same similarities, different temperatures
similarities = np.array([0.9, 0.5, 0.3, 0.1])  # Image vs 4 text categories
temps = [0.01, 0.05, 0.1, 0.5, 1.0]

print("Raw cosine similarities: ", similarities)
print(f"{'':>5}(correct match has similarity 0.9)\n")

fig, axes = plt.subplots(1, len(temps), figsize=(16, 4), sharey=True)

for ax, temp in zip(axes, temps):
    logits = similarities / temp
    probs = softmax(logits)
    
    bars = ax.bar(range(4), probs, color=['#4CAF50', '#FFC107', '#FF9800', '#F44336'])
    ax.set_title(f'\u03c4 = {temp}', fontsize=13, fontweight='bold')
    ax.set_xticks(range(4))
    ax.set_xticklabels(['0.9\n(match)', '0.5', '0.3', '0.1'], fontsize=9)
    ax.set_ylim(0, 1.1)
    
    for bar, p in zip(bars, probs):
        if p > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{p:.2f}', ha='center', fontsize=9)

axes[0].set_ylabel('Probability', fontsize=12)
fig.suptitle('Effect of Temperature on Softmax Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Small τ (0.01): nearly all probability on the correct match (too sharp — ignores hard negatives)")
print("Large τ (1.0):  probability spread across all options (too flat — cannot discriminate)")
print("CLIP uses τ ≈ 0.07: sharp enough to discriminate, soft enough to learn from hard negatives")

In [ ]:
# Show how temperature affects gradient magnitude
# The gradient of the loss with respect to the correct similarity tells us
# how strongly the model learns from each example

correct_sim = 0.8  # Similarity for the matching pair
wrong_sims = np.array([0.5, 0.3, 0.1])  # Similarities for non-matching pairs

temps_fine = np.linspace(0.01, 1.0, 100)
grad_magnitudes = []

for temp in temps_fine:
    all_sims = np.concatenate([[correct_sim], wrong_sims])
    logits = all_sims / temp
    probs = softmax(logits)
    # Gradient of cross-entropy loss w.r.t. logit of correct class
    # = -(1 - p_correct) / temp
    grad = (1 - probs[0]) / temp
    grad_magnitudes.append(grad)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(temps_fine, grad_magnitudes, color='#2196F3', linewidth=2.5)

# Mark the CLIP default
clip_idx = np.argmin(np.abs(temps_fine - 0.07))
ax.scatter([0.07], [grad_magnitudes[clip_idx]], color='red', s=100, zorder=5, label='CLIP default (τ=0.07)')

ax.set_xlabel('Temperature (τ)', fontsize=13)
ax.set_ylabel('Gradient Magnitude', fontsize=13)
ax.set_title('Gradient Magnitude vs Temperature\n(too high at small τ = unstable, too low at large τ = slow learning)', fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("At very small τ: gradients explode → training is unstable")
print("At very large τ: gradients vanish → training is very slow")
print("CLIP learns τ as a parameter, starting at ~0.07, to find the balance.")

## Summary

In this notebook, we built the core mechanics of CLIP from scratch:

| Step | What We Did | Key Insight |
|------|------------|-------------|
| 1. Cosine similarity | Measured embedding direction agreement | Only direction matters, not magnitude |
| 2. Contrastive learning | Trained with InfoNCE loss on matching pairs | Push matches together, push non-matches apart |
| 3. Embedding visualization | Plotted learned space in 2D | Matching image-text pairs cluster together |
| 4. Zero-shot classification | Classified images using only text descriptions | No labeled training data needed |
| 5. Temperature scaling | Controlled softmax sharpness with τ | Too small = unstable, too large = no discrimination |

### What Real CLIP Adds

Our simplified version used linear projections. Real CLIP uses:
- A **Vision Transformer (ViT)** as the image encoder (processes image patches through 12+ transformer layers)
- A **text Transformer** as the text encoder (processes tokens through 12+ transformer layers)
- Training on **400 million image-text pairs** scraped from the internet
- Batch size of **32,768** for enough hard negatives
- **Learned temperature** that adapts during training

But the core idea is exactly what we built here: map images and text to the same space, train with contrastive loss, and compare with cosine similarity.

**Next:** For failure modes, complexity benchmarks, and ablation experiments, see [01_vision_language_experiments.ipynb](./01_vision_language_experiments.ipynb). For interview-depth math and Q&A, see [vision-language-interview.md](./vision-language-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)